# Reproduce NICEDT-2027 concrete-crack analysis

This notebook re-analyzes saved SDNET2018 predictions; it does not retrain a CNN or rerun inference. Upload `sdnet2018_external_predictions_sanitized.csv` from the research package when prompted. GPU is not required.


In [ ]:
from google.colab import files
from pathlib import Path
import subprocess, sys, shutil, os

uploaded = files.upload()  # select sdnet2018_external_predictions_sanitized.csv
csv_candidates = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_candidates) != 1:
    raise ValueError(f'Upload exactly one prediction CSV; found: {csv_candidates}')
uploaded_csv = Path('/content') / csv_candidates[0]

repo = Path('/content/crack-detector')
if not (repo / 'research/nicedt2027-operating-point-analysis/src/analyze_predictions.py').exists():
    subprocess.run(['git','clone','https://github.com/aersews/crack-detector.git',str(repo)],check=True)
study = repo / 'research/nicedt2027-operating-point-analysis'
data_dir = study / 'data'
data_dir.mkdir(parents=True,exist_ok=True)
input_csv = data_dir / 'sdnet2018_external_predictions_sanitized.csv'
shutil.copy2(uploaded_csv,input_csv)
print('Repository:', repo)
print('Prediction CSV:', input_csv)


In [ ]:
study = Path('/content/crack-detector/research/nicedt2027-operating-point-analysis')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(study/'requirements.txt')],check=True)
results_dir = study / 'results_reproduced'
figures_dir = study / 'figures_reproduced'
subprocess.run([
    sys.executable, str(study/'src/analyze_predictions.py'),
    '--input', str(study/'data/sdnet2018_external_predictions_sanitized.csv'),
    '--output-dir', str(results_dir), '--figure-dir', str(figures_dir),
    '--bootstrap','2000','--seed','20261019'
],check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Image
print('Overall metrics')
display(pd.read_csv(results_dir/'overall_metrics.csv'))
print('Surface-specific metrics')
display(pd.read_csv(results_dir/'surface_metrics.csv'))
print('Patch vs group uncertainty comparison')
display(pd.read_csv(results_dir/'patch_vs_source_bootstrap_comparison.csv'))
display(Image(filename=str(figures_dir/'surface_recall_with_group_bootstrap_ci.png')))


## Interpretation limits
- The threshold is fixed at 0.31; do not tune it using SDNET2018 labels and then report the same data as an untouched test set.
- Group IDs come from numeric filename prefixes. This script checks the naming convention but cannot prove each group is an independent parent photograph.
- The original model checkpoint and inference pipeline are not included or independently verified. Results reproduce the analysis of the saved prediction file only.
